In [0]:
%run "../libs/libs"

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col

In [0]:
order_df = spark.read.format("delta").table("dev.tracking_sl.order")

In [0]:
invoice_df = spark.read.format("delta").table("dev.tracking_sl.invoice")

In [0]:
sender_df = order_df.select(col('sendername').alias('name'),col('sendercity').alias('city'),col('senderstate').alias('state')).distinct()

In [0]:
receiver_df = order_df.select(col('receivername').alias('name'),col('receivercity').alias('city')\
    ,col('receiverstate').alias('state')).distinct()


In [0]:
customer_df = invoice_df.select(col('nameofcustomer').alias('name'),col('invoicecity').alias('city')\
    ,col('invoicestate').alias('state')).distinct()


In [0]:
sender_df = sender_df.union(receiver_df)

In [0]:
sender_df = sender_df.union(customer_df)

In [0]:
sender_df = sender_df.withColumn('ingestion_time', current_timestamp())

In [0]:
sender_df.createOrReplaceTempView("temp_org")

In [0]:
%sql
merge into dev.tracking_gd.dimorganization tgt
USING ( 
  SELECT DISTINCT * 
  FROM temp_org 
) src 
on tgt.name = src.name
  and tgt.city = src.city
  and tgt.state = src.state
when not matched then
  insert (name, city, state, ingestion_time)
  Values(src.name,src.city,src.state,src.ingestion_time)

In [0]:
%sql
select * From dev.tracking_gd.dimorganization